# Stretch: more pandas, and more cleaning, worked solutions

The answers to the stretch notebook. Try each one before looking.

---

For when the live exercises are done and there is time left. These get
harder as you go, so nobody is expected to finish them all. Stretches 1 to
4 build on the first half of the session, 5 to 7 on the cleaning.

Each one adds one idea, and the notebook tells you what it is when you get
there. Where a stretch has `assert` lines, uncomment them once your answer
exists: they are the real numbers, so they tell you when you are right.

Four tables are loaded: `plays` and `artists`, the clean CSVs, which
already carry the artist's name, genre and country on every play; `raw`,
the messy file exactly as `load_messy` returns it; and `cleaned`, the
result of `clean_plays(raw)`.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

sys.path.insert(0, "sessions/session-06/solutions")
from cleaning import clean_plays, load_messy

plays = pd.read_csv("data/clean/plays.csv")
artists = pd.read_csv("data/clean/artists.csv")

raw = load_messy("data/messy/plays_messy.csv")
cleaned = clean_plays(raw)

print(plays.shape, artists.shape, raw.shape, cleaned.shape)

## Stretch 1: merging on a name

In [ ]:
clash = plays.merge(artists, on="artist_name")
print(clash.columns.tolist())

In [ ]:
merged = plays.merge(
    artists[["artist_name", "formed_year"]],
    on="artist_name",
    validate="many_to_one",
)

assert len(merged) == len(plays)
assert "formed_year" in merged.columns
assert "genre_x" not in merged.columns
print(len(plays), "->", len(merged), "rows")

Picking the columns before the merge avoids the clash entirely, and makes
the merge say what it is for: "bring me each play's formed year".

If you really do want both versions of a column, `suffixes=("_play",
"_artist")` gives them better names than `_x` and `_y`. Here they agree
anyway, so there is no point keeping both.

`validate="many_to_one"` passes because every artist name appears once in
`artists.csv`. That is worth checking whenever you join on a **name** rather
than an ID, because names are not guaranteed to be unique. Session 4's
stretch found three different artists with a track called *Paper Signal*.
Merging on track name would have fanned out, and `validate` would have said
so.

## Stretch 2: decades, and the plays that go missing

In [ ]:
merged["decade"] = merged["formed_year"] // 10 * 10

per_decade = merged.groupby("decade").size()

assert per_decade.loc[2010] == 1087
per_decade

In [ ]:
print(per_decade.sum(), "plays counted, out of", len(plays))

2,130, not 2,183. **53 plays have vanished, without an error.**

They belong to Rosewood Lane and Cassette Revival, the two artists with no
`formed_year`. Their year is `NaN`, so their decade is `NaN` too, because any
sum involving a missing number is missing. And `groupby` **drops missing
keys by default**. The rows were not filtered by you; they were quietly left
out of every group.

This is the same decision as the session 4 stretch: skip, pretend, or label.
`groupby` picked "skip" for you, and did not mention it. The fix is to ask
for the missing group explicitly:

In [ ]:
per_decade_all = merged.groupby("decade", dropna=False).size()

assert per_decade_all.sum() == len(plays)
assert per_decade_all.isna().sum() == 0               # the counts are all real numbers
assert per_decade_all[per_decade_all.index.isna()].iloc[0] == 53
per_decade_all

Now the `NaN` row is there with its 53 plays, and the total is right. The
last two lines check different things: the **counts** are never missing,
but one of the **labels** is.

Also note the decades are `2010.0`, not `2010`: a column with a `NaN` in it
is stored as decimals, as question 3 of the session 5 homework showed.

**The habit:** after a `groupby`, check that the groups add up to the
total. It is the same habit as checking the row count after a merge, and it
catches the same kind of silent loss.

## Stretch 3: who is in one table and not the other?

In [ ]:
never = artists[~artists["artist_name"].isin(plays["artist_name"])]["artist_name"]

assert set(never) == {"Iron Fernway", "Gravel Choir"}
never

`~` turns every `True` into `False` and back, so `~...isin(...)` means "not
in". It is `NOT IN` from SQL.

In [ ]:
played_names = plays[["artist_name"]].drop_duplicates()

found = artists.merge(played_names, on="artist_name", how="left", indicator=True)

assert len(found) == len(artists)
assert found["_merge"].value_counts()["left_only"] == 2
found["_merge"].value_counts()

In [ ]:
found[found["_merge"] == "left_only"][["artist_name", "genre", "country"]]

`indicator=True` does a left join and labels every row with where it was
found. `"left_only"` is the `WHERE p.play_id IS NULL` from session 5, and
it has the advantage that you can see the matched ones too, all in one
table.

In [ ]:
fanned = artists.merge(plays[["artist_name"]], on="artist_name", how="left", indicator=True)
print(len(artists), "artists ->", len(fanned), "rows with the whole of plays")

With the whole of `plays` on the right, each artist meets every one of their
plays: 2,185 rows, the 2,183 plays plus the two unmatched artists. That is
fan-out again, in a merge whose only job was to answer yes or no. Reducing
the right side to one row per artist first (`drop_duplicates`) is the same
"aggregate first" fix as the session, in its smallest form.

## Stretch 4: the top artist in each genre

In [ ]:
pairs = plays.groupby(["genre", "artist_name"]).agg(
    plays=("play_id", "count"),
    minutes=("minutes_played", "sum"),
).reset_index()

top = pairs.sort_values("minutes", ascending=False).groupby("genre").head(1)

assert top.set_index("genre").loc["Electronic", "artist_name"] == "Glass Tram"
assert len(top) == 8
top.round(1)

`groupby("genre").head(1)` keeps the first row of each group **in the order
the rows are already in**. That is why the sort comes first: it decides
what "first" means.

In [ ]:
top_two = pairs.sort_values("minutes", ascending=False).groupby("genre").head(2)
top_two.sort_values(["genre", "minutes"], ascending=[True, False]).round(1)

In [ ]:
electronic = top_two[top_two["genre"] == "Electronic"]["minutes"]
gap = electronic.iloc[0] - electronic.iloc[1]

assert round(gap, 1) == 5.8
print(f"Electronic: Glass Tram wins by {gap:.1f} minutes out of about 1,170")

**Electronic is the genre where the winner is barely a winner.** Glass Tram
beats DJ Kompas by under six minutes, out of nearly 1,200 each: about one
play's worth. In every other genre the gap is at least 17 minutes, and
usually far more.

By plays it is Rock that becomes a coin toss:

In [ ]:
by_plays = pairs.sort_values("plays", ascending=False).groupby("genre").head(2)
rock = by_plays[by_plays["genre"] == "Rock"][["artist_name", "plays"]]

assert list(rock["plays"]) == [64, 63]
rock

The Quiet Wires 64, Static Meadow 63. One play.

The same lesson as the tie in the session 4 homework and the busiest day in
session 5: a ranking always produces a winner, whether or not there really
is one. `head(2)` instead of `head(1)` costs nothing and tells you which
kind you have.

## Stretch 5: a damage report

In [ ]:
def damage_report(df):
    """Return one row per column: blanks, distinct values, and distinct values after strip and lower."""
    rows = []
    for column in df.columns:
        text = df[column].astype(str)
        rows.append({
            "column": column,
            "blanks": (text == "").sum(),
            "spellings": text.nunique(),
            "after strip and lower": text.str.strip().str.lower().nunique(),
        })
    return pd.DataFrame(rows)


report = damage_report(raw)

assert len(report) == 9
assert report.set_index("column").loc["device", "spellings"] == 21
assert report.set_index("column").loc["device", "after strip and lower"] == 6
assert report.set_index("column").loc["genre", "blanks"] == 36

report

A list of dictionaries, one per column, becomes a DataFrame in one call.
The keys become the column names. That is the session 4 shape, and now you
have built a table with it instead of reading one.

Reading the report:

* **the most spellings is `play_id`, with 2,183**, and that is not damage
  at all. An id is supposed to be different on every row. A number of
  spellings only means something next to what you expected
* **genre and device** drop sharply after strip and lower: 33 to 9, and 21
  to 6. That gap is the damage. (9 and 6, not 8 and 5, because the blank
  counts as a value.)
* **`minutes played `** barely changes, because the problem there is not
  capitals. The report does not catch `" min"`. No single check catches
  everything, which is why notebook 04 looked at each column by eye too
* **country has no blanks**. On the default read in notebook 04,
  `isna().sum()` said 49 missing. Those were the `"NA"` rows, which this
  report sees as an ordinary two-letter value, because `load_messy` keeps
  text as text

## Stretch 6: what were the unknowns?

In [ ]:
merged = cleaned.merge(plays, on="play_id", suffixes=("_ours", "_clean"), validate="one_to_one")
assert len(merged) == 2183

unknowns = merged[merged["device_ours"] == "unknown"]
assert len(unknowns) == 69

comparison = pd.DataFrame({
    "unknowns were": unknowns["device_clean"].value_counts(),
    "share of unknowns": unknowns["device_clean"].value_counts(normalize=True).round(3),
    "share of all plays": plays["device"].value_counts(normalize=True).round(3),
})
comparison

35 phones, 14 laptops, 10 speakers, 6 cars, 4 tablets. Phones are 51% of
the unknowns and 48% of all plays; every device is within a few points of
its share. The blanks hit every device about evenly.

**Why that matters.** If the blanks are spread evenly, dropping them or
labelling them barely changes each device's share, so plays per device
stays fair either way. If the export had blanked out, say, mostly car
plays, then dropping the blanks would quietly make the car look less used
than it was. Missing at random and missing for a reason need different
treatment, and you can only tell them apart by looking.

In real work you have no clean file to merge with. You can still compare
the rows with blanks against the rest on columns you **do** have: their
genres, their dates, their skip rate.

In [ ]:
print(f"skip rate of the unknowns: {unknowns['skipped_ours'].mean():.1%}")
print(f"skip rate of everything:   {cleaned['skipped'].mean():.1%}")

15.9% against 11.8%. That is 11 skips out of 69, where the overall rate
would predict about 8. With a group this small, three extra skips is the
kind of difference chance produces all the time, which is one more reason
not to rank `"unknown"` against the real devices.

## Stretch 7: guess the missing minutes, and get marked

In [ ]:
TRUTH = plays["minutes_played"].sum()
minutes = cleaned["minutes_played"]

by_artist = cleaned.groupby("artist_name")["minutes_played"].transform("median")
by_artist_and_skip = cleaned.groupby(["artist_name", "skipped"])["minutes_played"].transform("median")


def imputed_total(filled):
    """Return the total of a minutes column after its gaps have been filled."""
    return filled.sum()


methods = {
    "leave missing": minutes,
    "average of all plays": minutes.fillna(minutes.mean()),
    "artist median": minutes.fillna(by_artist),
    "artist and skipped median": minutes.fillna(by_artist_and_skip),
}

results = pd.DataFrame([
    {"method": name, "total": round(imputed_total(filled), 1),
     "off by": round(imputed_total(filled) - TRUTH, 1)}
    for name, filled in methods.items()
])

assert round(minutes.sum(), 1) == 8608.0
assert round(TRUTH, 1) == 8980.4
assert by_artist_and_skip[minutes.isna()].notna().all()     # every gap got a guess
results

**1. Why is the artist median worse than the plain average?** Because of
who the missing plays are.

In [ ]:
print(f"skipped, among the missing:  {cleaned.loc[minutes.isna(), 'skipped'].mean():.1%}")
print(f"skipped, among all plays:    {cleaned['skipped'].mean():.1%}")

16.0% of the missing plays were skipped, against 11.8% overall, and a
skipped play is short. The artist median is the length of a typical
**full** play by that artist, so for every skipped play it guesses several
minutes too many. Using more information made it worse, because it was the
wrong information. Adding `skipped` to the group fixes exactly that, and
lands within 3.4 minutes of the truth.

**2. Is leaving them missing still the most honest?** It has the biggest
error, 372.4 minutes short, and it is the only total that is a fact: it is
the exact total of the 2,089 plays whose length we know. Every other row
is an estimate. So the label is what matters:

* "8,608.0 minutes, from the 2,089 plays with a recorded length" is true
* "about 8,977 minutes, with 94 play lengths estimated" is honest
* "8,977.0 minutes", with no note, is a guess dressed up as a measurement

**3. With no clean file, how do you choose?** You check the assumption
behind each method against the data you have, as question 1 did: are the
missing rows like the others? Here they were skipped more often, which
rules out any method that ignores `skipped`. And you say what you did.
An estimate that is labelled is a tool; one that is not is a mistake
waiting for someone to find it.